# 03 · Nettoyage de la table CIQUAL 2020 (C3)

**Objectif :** produire un jeu d'aliments propre, prêt pour l'import PostgreSQL, à partir du fichier brut CIQUAL.

**Principe directeur :** une entrée *corrompue* (inutilisable) n'est pas une entrée *non scorable*.
Les règles de non-calcul du score (complétude < 50 %, apport < 1 g) relèvent du calcul à la volée,
pas du nettoyage. Un aliment sans score reste en base pour que l'application affiche « Score indisponible ».

**Source :** Anses. 2020. Table de composition nutritionnelle des aliments Ciqual. Licence Ouverte / Etalab 2.0.
Version 2020 conservée volontairement : le référentiel de score v1.0 est calibré dessus (Ciqual 2025 publiée en 11/2025, migration = recalibration v2.0).

--------------------------
## 0.1 Imports, chemins et constantes

In [ ]:
from pathlib import Path
import re

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

CIQUAL_VERSION = "2020"   # ne pas changer sans recalibrer le référentiel de score (v1.0 calibré sur 2020)
CIQUAL_XLS_URL = "https://www.data.gouv.fr/api/1/datasets/r/bcdb7fec-875c-42aa-ba6e-460adf97aad3"
RAW_FILE  = Path("data/raw") / f"ciqual_{CIQUAL_VERSION}.xls"
CLEAN_DIR = Path("data/clean")
CITATION  = "Anses. 2020. Table de composition nutritionnelle des aliments Ciqual."

In [ ]:
# Téléchargement 
def download_ciqual(url: str, dest: Path, force: bool = False) -> Path:
    """Télécharge le fichier CIQUAL s'il n'est pas déjà en cache local. 
    (ne retélécharge pas si le fichier existe déjà (sauf force=True).)
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not force:
        print(f"Déjà présent : {dest} ({dest.stat().st_size/1_000_000:.1f} Mo)")
        return dest
    print(f"Téléchargement CIQUAL depuis data.gouv.fr…")
    with requests.get(url, stream=True, timeout=60,
                      headers={"User-Agent": "Alimendo-datacollect/1.0"}) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"Enregistré : {dest} ({dest.stat().st_size/1_000_000:.1f} Mo)")
    return dest

## 0.2 Chargement BRUT : aucune conversion, tout reste du texte
Différence avec load_ciqual() du notebook 01 :   
on force dtype=str et on désactive la détection automatique des NaN.   
**Objectif :** voir les valeurs EXACTEMENT comme dans la source ('-', 'traces', '< 0,1', '12,5'…) avant de décider comment les convertir.


In [ ]:
def load_ciqual_brut(path: Path) -> tuple[pd.DataFrame, str]:
    """Charge le fichier CIQUAL sans aucune interprétation des valeurs.

    Retourne (DataFrame 100 % texte, format détecté).
    Lève ValueError si le format n'est pas reconnu (plutôt que de deviner).
    """
    head = path.read_bytes()[:8]
    opts = dict(dtype=str, keep_default_na=False, na_values=[])  # cellule vide -> '' (pas NaN)
    if head.startswith(b"PK\x03\x04"):                 # zip -> xlsx
        return pd.read_excel(path, engine="openpyxl", **opts), "xlsx"
    if head.startswith(b"\xD0\xCF\x11\xE0"):            # OLE2 -> xls Excel 97-2003
        return pd.read_excel(path, engine="xlrd", **opts), "xls (OLE2)"
    if head.lstrip()[:1] == b"<":                       # export HTML déguisé en .xls
        # thousands=None : sinon '12,5' serait lu comme 125 (séparateur de milliers par défaut)
        df = pd.read_html(path, thousands=None, keep_default_na=False)[0]
        return df.astype(str), "html déguisé en .xls"
    raise ValueError(f"Format CIQUAL non reconnu (premiers octets : {head!r})")


path = download_ciqual(CIQUAL_XLS_URL, RAW_FILE)
brut, fmt = load_ciqual_brut(path)
assert len(brut) > 0, "Le fichier CIQUAL est vide"

print(f"Source : {CITATION}")
print(f"Format détecté : {fmt}")
print(f"Dimensions brutes : {brut.shape[0]} lignes × {brut.shape[1]} colonnes")
print("Types pandas :", brut.dtypes.astype(str).value_counts().to_dict())  # attendu : 100 % texte
display(brut.head(3))

------------------------
## 1. Audit structurel : identifiants, doublons, groupes
### 1.1 Identifiants et libellés : vides, unicité, format

In [ ]:
cols_ident = ["alim_code", "alim_nom_fr", "alim_grp_code", "alim_grp_nom_fr",
              "alim_ssgrp_nom_fr", "alim_ssssgrp_nom_fr"]

ident_txt = brut[cols_ident].fillna("").apply(lambda s: s.str.strip())

audit_ident = pd.DataFrame({
    "vides":              (ident_txt == "").sum(),
    "tiret_seul":         (ident_txt == "-").sum(),
    "valeurs_distinctes": ident_txt.nunique(),
})
print("Tableau 1.1 — Complétude et unicité des colonnes d'identification")
display(audit_ident)

# alim_code doit être un entier unique : c'est la seule clé naturelle de la source
code_non_numerique = ~ident_txt["alim_code"].str.fullmatch(r"\d+")
print(f"alim_code non numériques : {code_non_numerique.sum()}")
print(f"alim_code dupliqués      : {ident_txt['alim_code'].duplicated(keep=False).sum()} lignes")
print(f"alim_nom_fr dupliqués    : {ident_txt['alim_nom_fr'].duplicated(keep=False).sum()} lignes")

### 1.2 Lignes à examiner : noms dupliqués et aliments sans groupe

In [ ]:
noms_dupliques = brut.loc[ident_txt["alim_nom_fr"].duplicated(keep=False),
                          ["alim_code", "alim_nom_fr", "alim_grp_nom_fr", "alim_ssgrp_nom_fr"]]
print(f"Tableau 1.2a — Noms dupliqués ({len(noms_dupliques)} lignes)")
display(noms_dupliques.sort_values("alim_nom_fr"))

sans_groupe = brut.loc[ident_txt["alim_grp_nom_fr"].isin(["", "-"]),
                       ["alim_code", "alim_grp_code", "alim_nom_fr"]]
print(f"Tableau 1.2b — Aliments sans groupe ({len(sans_groupe)} lignes)")
display(sans_groupe)

print("Tableau 1.2c — Répartition par groupe (code + nom)")
display(brut.groupby(["alim_grp_code", "alim_grp_nom_fr"]).size()
            .rename("n_aliments").reset_index())

### Lecture § 1.1–1.2
- 3 185 codes distincts : un seul doublon, `alim_code` 9621 « Son de blé », présent dans deux sous-groupes.
- 45 lignes sans nom de groupe :
  - 44 de code `03` (farines, amidons, pâtes) : nom vide alors que 145 autres lignes `03` portent « produits céréaliers » → **format non normalisé** dans la source, pas une entrée corrompue (le code est présent).
  - 1 de code `00` « Dessert (aliment moyen) » : agrégat sans groupe.
- `alim_ssssgrp_nom_fr` : 1 259 « - » = absence de sous-sous-groupe → NULL au § 3.
-----------------------
### 1.3 Doublon alim_code 9621 « Son de blé » : même fiche ou deux fiches différentes ?


In [ ]:
doublon_code = brut[ident_txt["alim_code"].duplicated(keep=False)]
assert len(doublon_code) == 2

cols_qui_different = [c for c in brut.columns if doublon_code[c].nunique() > 1]
print(f"Colonnes qui diffèrent entre les 2 lignes : {len(cols_qui_different)} / {brut.shape[1]}")
print("Tableau 1.3 — Valeurs qui diffèrent (une ligne par colonne)")
display(doublon_code[cols_qui_different].T)

### 1.4 Aliments sans nom de groupe : peut-on récupérer le nom depuis le code ?


In [ ]:
sans_nom_grp = ident_txt["alim_grp_nom_fr"] == ""
cols_codes = ["alim_grp_code", "alim_ssgrp_code", "alim_ssssgrp_code",
              "alim_ssgrp_nom_fr", "alim_ssssgrp_nom_fr"]

print("Tableau 1.4a — Codes de (sous-)groupe des 45 aliments sans nom de groupe")
display(brut.loc[sans_nom_grp, cols_codes].value_counts().rename("n").reset_index())

# Condition pour récupérer le nom depuis le code : chaque code groupe ne correspond qu'à UN nom
lignes_nommees = brut.loc[~sans_nom_grp]
noms_par_code = lignes_nommees.groupby("alim_grp_code")["alim_grp_nom_fr"].nunique()
print("Codes groupe associés à plus d'un nom :", noms_par_code[noms_par_code > 1].to_dict() or "aucun")

print("Tableau 1.4b — Sous-groupes nommés existants dans le groupe 03")
display(lignes_nommees[lignes_nommees["alim_grp_code"].str.strip() == "03"]
        .groupby(["alim_ssgrp_code", "alim_ssgrp_nom_fr"]).size()
        .rename("n").reset_index())

### 1.5 La ligne dont le sous-groupe vaut « - »

In [ ]:
print("Tableau 1.5 — Aliment avec sous-groupe « - »")
display(brut.loc[ident_txt["alim_ssgrp_nom_fr"] == "-",
                 ["alim_code", "alim_nom_fr", "alim_grp_code", "alim_grp_nom_fr",
                  "alim_ssgrp_code", "alim_ssgrp_nom_fr"]])

### 1.6 Doublon 9621 : les deux lignes se contredisent-elles ou se complètent-elles ?

In [ ]:
cols_nutri = [c for c in brut.columns if not c.startswith("alim_")]
print(f"Colonnes nutritionnelles : {len(cols_nutri)}")

VIDES = {"", "-"}
i1, i2 = doublon_code.index
v1 = doublon_code.loc[i1, cols_nutri].fillna("").str.strip()
v2 = doublon_code.loc[i2, cols_nutri].fillna("").str.strip()

def compare(a: str, b: str) -> str:
    """Compare deux valeurs brutes d'une même colonne."""
    a_ok, b_ok = a not in VIDES, b not in VIDES
    if a_ok and b_ok:
        return "identiques" if a == b else "CONFLIT"
    if a_ok:
        return f"seulement ligne {i1}"
    if b_ok:
        return f"seulement ligne {i2}"
    return "vide des deux côtés"

comparaison_9621 = pd.DataFrame({f"ligne_{i1}": v1, f"ligne_{i2}": v2})
comparaison_9621["statut"] = [compare(a, b) for a, b in zip(v1, v2)]

print("Tableau 1.6a — Bilan colonne par colonne")
display(comparaison_9621["statut"].value_counts().rename("n_colonnes").to_frame())
print("Tableau 1.6b — Colonnes en conflit (valeurs renseignées des deux côtés mais différentes)")
display(comparaison_9621[comparaison_9621["statut"] == "CONFLIT"])

## 2. Inventaire des formats non normalisés

----------------------
### 2.1 Classer chaque cellule nutritionnelle selon son format brut
Mesure sur TOUT le jeu (toutes les colonnes, pas seulement les 23 du score) :
c'est l'« identification des entrées au format non normalisé » demandée en C3.

In [ ]:

FORMATS = ["nombre (point)", "nombre (virgule)", "vide", "tiret",
           "traces", "inférieur à (< X)", "autre texte"]

def classer_format(valeur) -> str:
    """Retourne le format d'une valeur brute CIQUAL (aucune conversion)."""
    v = "" if pd.isna(valeur) else str(valeur).replace("\xa0", " ").strip()
    if v == "":                                         return "vide"
    if v == "-":                                        return "tiret"
    if v.lower() == "traces":                           return "traces"
    if v.startswith("<"):                               return "inférieur à (< X)"
    if re.fullmatch(r"-?\d+(\.\d+)?([eE]-?\d+)?", v):   return "nombre (point)"   # cellule Excel numérique (dont 1e-05)
    if re.fullmatch(r"-?\d+,\d+", v):                   return "nombre (virgule)" # texte à la française
    return "autre texte"

# Garde-fou : la fonction sur des cas limites, avant de l'appliquer à 200 000 cellules
cas_limites = ["", " - ", "Traces", "< 0,1", "<0.5", "12", "12.5", "12,5", "-3",
               "1e-05", "\xa012,5", "1 200", "#REF!", None]
display(pd.DataFrame({"brut": [repr(c) for c in cas_limites],
                      "format": [classer_format(c) for c in cas_limites]}))

formats_cellules = brut[cols_nutri].apply(lambda col: col.map(classer_format))
inventaire_formats = (formats_cellules.apply(lambda col: col.value_counts()).T
                      .reindex(columns=FORMATS).fillna(0).astype(int))
inventaire_formats.index.name = "colonne CIQUAL"
assert inventaire_formats.sum(axis=1).eq(len(brut)).all(), "Chaque colonne doit compter toutes les lignes"

total_formats = inventaire_formats.sum().rename("n_cellules").to_frame()
total_formats["%"] = (100 * total_formats["n_cellules"] / total_formats["n_cellules"].sum()).round(2)
print(f"Tableau 2.1a — Formats sur l'ensemble des {len(cols_nutri)} colonnes × {len(brut)} aliments")
display(total_formats)

espaces_parasites = brut[cols_nutri].apply(
    lambda col: (col.fillna("") != col.fillna("").str.strip()) | col.fillna("").str.contains("\xa0")
).sum().sum()
print(f"Cellules avec espaces parasites (début/fin ou insécables) : {espaces_parasites}")

print("Tableau 2.1b — Valeurs « autre texte » rencontrées")
autres_textes = brut[cols_nutri].where(formats_cellules.eq("autre texte")).stack()
display(autres_textes.value_counts().rename("n").to_frame() if len(autres_textes) else "aucune")

### 2.2 Visualisation : part des valeurs non numériques par colonne
Une barre par colonne CIQUAL ; seules les valeurs NON numériques sont colorées,
le reste de la barre (jusqu'à 100 %) correspond aux valeurs numériques.

In [ ]:

SEGMENTS = {"vide": "#2a78d6", "tiret": "#eb6834", "traces": "#1baf7a",
            "inférieur à (< X)": "#eda100", "autre texte": "#e87ba4"}

part_formats = 100 * inventaire_formats.div(inventaire_formats.sum(axis=1), axis=0)
part_formats = part_formats.loc[part_formats[list(SEGMENTS)].sum(axis=1).sort_values().index]
etiquettes = [re.sub(r"\s*\([^()]*/100 ?g\)\s*$", "", c)[:45] for c in part_formats.index]  # retire l'unité pour la lisibilité

fig, ax = plt.subplots(figsize=(10, max(4, 0.22 * len(part_formats))))
gauche = np.zeros(len(part_formats))
for fmt, couleur in SEGMENTS.items():
    ax.barh(etiquettes, part_formats[fmt], left=gauche, color=couleur, label=fmt,
            height=0.75, edgecolor="white", linewidth=0.8)
    gauche += part_formats[fmt].values
ax.set_xlim(0, 100)
ax.set_xlabel("% des aliments (le reste de la barre = valeur numérique)")
ax.set_title(f"Valeurs non numériques par colonne CIQUAL {CIQUAL_VERSION} (n = {len(brut)} aliments)")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e5e4e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

### Lecture § 1.6 et 2.1–2.2
- **Doublon 9621 « Son de blé »** : 0 conflit. La ligne 3121 porte 63 valeurs, la ligne 3120 une seule absente de l'autre.
  -> Décision : **fusion** en une ligne (valeurs de 3121, complétées par 3120). Aucune perte, aucune valeur inventée.
- **Formats** (67 colonnes × 3 186 aliments) : nombres à point 21,8 % et à virgule 38,9 % mélangés dans les mêmes colonnes ;
  tiret 29,8 % ; « < X » 7,8 % ; traces 0,9 % ; cellule vide 0,7 % ; **0 texte inattendu, 0 espace parasite**.
- La cellule vide n'est pas documentée par l'ANSES (seul « - » l'est) → répartition à mesurer avant de décider.

### 2.3 Cellules vides : réparties sur tout le jeu ou concentrées sur quelques aliments ?

In [ ]:
vides_par_aliment = formats_cellules.eq("vide").sum(axis=1)
print("Tableau 2.3a — Nombre de cellules vides par aliment")
display(vides_par_aliment.value_counts().sort_index().rename("n_aliments").to_frame())

aliments_avec_vides = (brut.loc[vides_par_aliment > 0, ["alim_code", "alim_nom_fr", "alim_grp_code"]]
                       .assign(n_vides=vides_par_aliment[vides_par_aliment > 0]))
print(f"Tableau 2.3b — Aliments ayant au moins une cellule vide ({len(aliments_avec_vides)}), les 40 plus vides")
display(aliments_avec_vides.sort_values("n_vides", ascending=False).head(40))

# Cas extrême : aucune valeur exploitable sur les 67 colonnes
sans_aucune_valeur = formats_cellules.isin(["vide", "tiret"]).all(axis=1)
print(f"Aliments sans AUCUNE valeur nutritionnelle (67 colonnes vides ou « - ») : {sans_aucune_valeur.sum()}")

### 2.4 « < X » et « traces » : une seule graphie, convertible sans ambiguïté ?


In [ ]:
inferieurs = (brut[cols_nutri].where(formats_cellules.eq("inférieur à (< X)"))
              .stack().dropna().str.strip())
MOTIF_INFERIEUR = r"<\s*\d+([.,]\d+)?"          # « < » suivi d'un nombre, virgule ou point
bien_formes = inferieurs.str.fullmatch(MOTIF_INFERIEUR)
print(f"« < X » bien formés : {bien_formes.sum()} / {len(inferieurs)}")
print("Tableau 2.4a — « < X » mal formés (s'il y en a)")
display(inferieurs[~bien_formes].value_counts().rename("n").to_frame())

traces = (brut[cols_nutri].where(formats_cellules.eq("traces"))
          .stack().dropna().str.strip())
print("Tableau 2.4b — Graphies de « traces »")
display(traces.value_counts().rename("n").to_frame())

### Lecture § 2.3–2.4
- 117 aliments ont ≥ 1 cellule vide (1 à 66), **0 aliment sans aucune valeur** → les cellules vides ne signalent pas d'entrée corrompue.
  L'aliment le plus vide (66) est la ligne 3120 du doublon 9621, déjà traitée par fusion.
- 16 630 / 16 630 « < X » bien formés ; « traces » : une seule graphie (1 904).
- Décision : cellule vide **et** « - » → NULL (non mesuré, jamais 0), conformément à la doc ANSES et au référentiel v1.0.